In [ ]:
%load_ext autoreload
%autoreload 2
import os
import pandas as pd
import matplotlib.pyplot as plt

# Anchor to the project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

from src.config import SimConfig, EnvConfig
from src.utils.data_processing import load_and_cache_entire_fleet
from src.utils.evaluation import VoyageBenchmarker, print_markdown_table
from src.solvers import AugmentedHybridSDPSolver
from src.plants import AugmentedHybridPlant
from src.controllers import build_approach, AugmentedValueControl, AugmentedPolicyControl, AugmentedFCLockedControl

In [ ]:
# 1. Setup the Environment and Load Data
env = EnvConfig()
fleet_data = load_and_cache_entire_fleet(env)

# Using the full 11 days for a highly rigorous scientific average
exclude_days = [1,2,3]

In [ ]:
# 2. Define the Test Vector for Spatial Resolution
# We span from a very fine, computationally heavy grid (80kW) to a coarse, fast grid (240kW)
dp_test_values = range(50,255,25)

results_data = []

print("--- RUNNING SPATIAL RESOLUTION SENSITIVITY ---")

for dp in dp_test_values:
    print(f"\n[ Evaluating Grid: dP = {dp} kW ]")
    
    # Instantiate a new config for this specific dP. Dt is locked at 300s.
    config = SimConfig(
        dP=dp, 
        Dt=300, 
        N_Pd=6, 
        use_smart_grid=True,
        n_pack= 4,
        verbose=False 
    )
    
    benchmarker = VoyageBenchmarker(fleet_data, env, config, exclude_days)
    
    value_real_factory = build_approach(
        controller_cls=AugmentedValueControl, 
        plant_cls=AugmentedHybridPlant, 
        solver_cls=AugmentedHybridSDPSolver, 
        is_macro=False
    )
    
    # Run the benchmark
    report = benchmarker.run_leave_one_out(value_real_factory)
    avg_metrics = report.summary.loc['Average']
    
    # Log the metrics for the Pareto front
    results_data.append({
        'dP [kW]': dp,
        'Average Total Cost [$]': avg_metrics['Total Cost [$]'],
        'Offline Compute Time [s]': avg_metrics['Offline Compute Time [s]'],
        'Online Compute Time [s]': avg_metrics['Online Compute Time [s]']
    })

# Convert to DataFrame for easy viewing
df_results = pd.DataFrame(results_data).set_index('dP [kW]')

In [ ]:
print("\n--- SPATIAL RESOLUTION: SENSITIVITY SUMMARY ---")
print_markdown_table(df_results)

In [ ]:
# 3. Plot the L-Shaped Pareto Curve for the Article
fig, ax = plt.subplots(figsize=(9, 6))

# Extract data
compute_times = df_results['Offline Compute Time [s]'].values
total_costs = df_results['Average Total Cost [$]'].values
dp_labels = df_results.index.values

# Plot the curve
ax.plot(compute_times, total_costs, marker='o', linestyle='-', color='royalblue', linewidth=2.5, markersize=8)

# Annotate each point with its dP value
for i, dp in enumerate(dp_labels):
    ax.annotate(f"dP={int(dp)}", 
                (compute_times[i], total_costs[i]), 
                xytext=(10, 5), 
                textcoords='offset points',
                fontsize=10, 
                fontweight='bold', 
                color='darkslategray')

# Formatting for academic standards
ax.set_title("Pareto Frontier: Spatial Resolution vs. Offline Complexity", fontsize=14, fontweight='bold')
ax.set_xlabel("Offline Compute Time [Seconds]", fontsize=12)
ax.set_ylabel("Average Total Cost [$]", fontsize=12)
ax.grid(True, linestyle='--', alpha=0.6)

# Save and Show
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/pareto_spatial_resolution.png', dpi=300, bbox_inches='tight')
plt.show()

# Optional: Print the complexity proof
print(f"\nComplexity Check:")
print(f"Ratio of Compute Time (dP=80 vs dP=160): {compute_times[0] / compute_times[2]:.2f}x")
print(f"Theoretical O(1/dP^3) Ratio: {(160**3) / (80**3):.2f}x")